In [2]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class RecyclingRobotEnv(gym.Env):
    """
    A Reinforcement Learning environment for a Recycling Robot.
    The robot moves left/right to collect plastic, paper, or metal items falling from the top.
    """
    metadata = {"render_modes": ["human"], "render_fps": 4}

    def __init__(self):
        super(RecyclingRobotEnv, self).__init__()

        # Define grid size (5x5 matrix layout)
        self.grid_size = 5

        # Actions: 0 = Move Left, 1 = Stay, 2 = Move Right
        self.action_space = spaces.Discrete(3)

        # Observation space: [robot_x, item_x, item_y, item_type]
        # item_type: 0 = Plastic, 1 = Paper, 2 = Metal
        low = np.array([0, 0, 0, 0], dtype=np.int32)
        high = np.array([self.grid_size - 1, self.grid_size - 1, self.grid_size - 1, 2], dtype=np.int32)
        self.observation_space = spaces.Box(low=low, high=high, dtype=np.int32)

        # Environment state initialization
        self.robot_x = 0
        self.item_x = 0
        self.item_y = 0
        self.item_type = 0
        self.steps_taken = 0
        self.max_steps = 20

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Spawn robot at the center bottom row
        self.robot_x = self.grid_size // 2

        # Randomly spawn an item at the top row (y = 0)
        # Fixed: Gymnasium random generators use .integers() instead of .randint()
        self.item_x = self.np_random.integers(0, self.grid_size)
        self.item_y = 0
        self.item_type = self.np_random.integers(0, 3)

        self.steps_taken = 0

        return self._get_obs(), {}

    def _get_obs(self):
        return np.array([self.robot_x, self.item_x, self.item_y, self.item_type], dtype=np.int32)

    def step(self, action):
        # 1. Apply action to move robot
        if action == 0 and self.robot_x > 0:
            self.robot_x -= 1
        elif action == 2 and self.robot_x < self.grid_size - 1:
            self.robot_x += 1

        # 2. Advance the falling item downward
        self.item_y += 1
        self.steps_taken += 1

        reward = 0
        terminated = False

        # 3. Check for collection/miss at the bottom row (y = grid_size - 1)
        if self.item_y == self.grid_size - 1:
            terminated = True
            if self.robot_x == self.item_x:
                reward = 10  # Successfully recycled!
            else:
                reward = -5  # Missed item / Litter penalty

        # Small step penalty to encourage rapid collection
        reward -= 0.1

        # 4. Check max step constraint
        truncated = self.steps_taken >= self.max_steps

        return self._get_obs(), reward, terminated, truncated, {}

    def render(self):
        grid = np.full((self.grid_size, self.grid_size), ".", dtype=object)

        # Mark robot position (R)
        grid[self.grid_size - 1, self.robot_x] = "R"

        # Mark item position by its type (P = Plastic, A = Paper, M = Metal)
        item_chars = ["P", "A", "M"]
        if self.item_y < self.grid_size:
            grid[self.item_y, self.item_x] = item_chars[self.item_type]

        print("\n" + "="*20)
        for row in grid:
            print(" ".join(row))
        print("="*20)

# Example execution with random actions
if __name__ == "__main__":
    env = RecyclingRobotEnv()
    obs, info = env.reset()

    print("Starting simulation...")
    env.render()

    done = False
    while not done:
        random_action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(random_action)
        env.render()
        print(f"Action Taken: {random_action} | Reward Received: {reward:.1f}")
        done = terminated or truncated



Starting simulation...

. . M . .
. . . . .
. . . . .
. . . . .
. . R . .

. . . . .
. . M . .
. . . . .
. . . . .
. . . R .
Action Taken: 2 | Reward Received: -0.1

. . . . .
. . . . .
. . M . .
. . . . .
. . . R .
Action Taken: 1 | Reward Received: -0.1

. . . . .
. . . . .
. . . . .
. . M . .
. . . . R
Action Taken: 2 | Reward Received: -0.1

. . . . .
. . . . .
. . . . .
. . . . .
. . M . R
Action Taken: 1 | Reward Received: -5.1
